# 🔍 Grid Search - JurisClass AI (LegalBert-pt)

Este notebook realiza a busca em grade (Grid Search) para encontrar os melhores hiperparâmetros do modelo LegalBert-pt.

**Antes de rodar:**
1. Vá em `Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4`
2. Faça upload dos arquivos: `train_clean.csv`, `val_clean.csv`, `class_weights.json`

## 1. Instalar dependências

In [ ]:
!pip install transformers datasets accelerate -q

## 2. Upload dos arquivos do dataset

In [ ]:
from google.colab import files
print("Faça upload dos 3 arquivos: train_clean.csv, val_clean.csv, class_weights.json")
uploaded = files.upload()

## 3. Verificar GPU disponível

In [ ]:
import torch
print(f"GPU disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. Carregar dados e preparar datasets

In [ ]:
import pandas as pd
import numpy as np
import json
from torch import nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EvalPrediction
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import itertools
import gc

# Carregar dados
print("Carregando dados...")
df_train = pd.read_csv("train_clean.csv")
df_val = pd.read_csv("val_clean.csv")

with open("class_weights.json", "r") as f:
    class_weights_dict = json.load(f)

# Amostragem estratificada para o Grid Search (3000 textos)
GRID_SEARCH_SAMPLE = 3000
if len(df_train) > GRID_SEARCH_SAMPLE:
    df_train, _ = train_test_split(
        df_train,
        train_size=GRID_SEARCH_SAMPLE,
        stratify=df_train['document_type'],
        random_state=42
    )

print(f"Treino: {len(df_train)} textos | Validação: {len(df_val)} textos")
print(f"\nDistribuição por classe (treino):")
print(df_train['document_type'].value_counts())

# Criar label2id e id2label
classes = sorted(list(class_weights_dict.keys()))
label2id = {c: i for i, c in enumerate(classes)}
id2label = {i: c for i, c in enumerate(classes)}

# Converter pesos para tensor
weights_list = [class_weights_dict[c] for c in classes]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(weights_list, dtype=torch.float).to(device)

print(f"\nUsando device: {device}")
print(f"Classes: {classes}")
print(f"Pesos: {weights_list}")

## 5. Tokenização

In [ ]:
MODEL_NAME = "raquelsilveira/legalbertpt_fp"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class LegalDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

print("Tokenizando dados (isso pode levar alguns minutos)...")
train_encodings = tokenizer(df_train['body'].tolist(), truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(df_val['body'].tolist(), truncation=True, padding=True, max_length=512)

train_labels = [label2id[label] for label in df_train['document_type']]
val_labels = [label2id[label] for label in df_val['document_type']]

train_dataset = LegalDataset(train_encodings, train_labels)
val_dataset = LegalDataset(val_encodings, val_labels)

print(f"Tokenização concluída! Train: {len(train_dataset)} | Val: {len(val_dataset)}")

## 6. Definir Trainer customizado e métricas

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=1)
    acc = accuracy_score(p.label_ids, preds)
    f1 = f1_score(p.label_ids, preds, average='macro')
    return {"accuracy": acc, "f1_macro": f1}

print("Trainer e métricas definidos!")

## 7. 🚀 Executar Grid Search

**36 combinações** de hiperparâmetros. Tempo estimado no Colab T4: **~3 a 4 horas**.

In [ ]:
# Espaço de busca completo
epochs_list = [1, 3, 5]
batch_size_list = [8, 16, 32]
lr_list = [2e-5, 5e-5]
warmup_list = [100, 500]

combinations = list(itertools.product(epochs_list, batch_size_list, lr_list, warmup_list))
print(f"Total de combinações: {len(combinations)}")
print(f"Combinações: epochs={epochs_list}, batch={batch_size_list}, lr={lr_list}, warmup={warmup_list}")

results = []

for i, (ep, bs, lr, wu) in enumerate(combinations):
    print(f"\n{'='*60}")
    print(f"--- Iteração {i+1}/{len(combinations)} ---")
    print(f"Epochs: {ep} | Batch: {bs} | LR: {lr} | Warmup: {wu}")
    print(f"{'='*60}")

    # Recarregar o modelo do zero
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(classes),
        id2label=id2label,
        label2id=label2id
    ).to(device)

    training_args = TrainingArguments(
        output_dir=f"./results_gs/ep{ep}_bs{bs}_lr{lr}_wu{wu}",
        num_train_epochs=ep,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs,
        learning_rate=lr,
        warmup_steps=wu,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        disable_tqdm=False,
        report_to="none",
        fp16=True,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    # Avaliar no conjunto de validação
    eval_metrics = trainer.evaluate()

    results.append({
        "epochs": ep,
        "batch_size": bs,
        "learning_rate": lr,
        "warmup_steps": wu,
        "val_loss": eval_metrics.get("eval_loss", 0),
        "val_accuracy": eval_metrics.get("eval_accuracy", 0),
        "val_f1_macro": eval_metrics.get("eval_f1_macro", 0)
    })

    print(f"\n✅ Resultado: Loss={eval_metrics.get('eval_loss', 0):.4f} | Acc={eval_metrics.get('eval_accuracy', 0):.4f} | F1={eval_metrics.get('eval_f1_macro', 0):.4f}")

    # Liberar memória da GPU
    del model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

print("\n\n🎉 GRID SEARCH FINALIZADO!")

## 8. Resultados

In [ ]:
# Salvar e exibir resultados
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="val_f1_macro", ascending=False)
df_results.to_csv("grid_search_results.csv", index=False)

print("=" * 60)
print("TOP 5 MELHORES COMBINAÇÕES (por F1 Macro):")
print("=" * 60)
print(df_results.head().to_string(index=False))

print(f"\n🏆 MELHOR CONFIGURAÇÃO:")
best = df_results.iloc[0]
print(f"  Epochs: {int(best['epochs'])}")
print(f"  Batch Size: {int(best['batch_size'])}")
print(f"  Learning Rate: {best['learning_rate']}")
print(f"  Warmup Steps: {int(best['warmup_steps'])}")
print(f"  F1 Macro: {best['val_f1_macro']:.4f}")
print(f"  Accuracy: {best['val_accuracy']:.4f}")
print(f"  Loss: {best['val_loss']:.4f}")

## 9. Download dos resultados

In [ ]:
from google.colab import files
files.download('grid_search_results.csv')
print("Download do CSV de resultados iniciado!")